In [1]:
import pandas as pd
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

# 1. Tu FeatureAdder
class FeatureAdder(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X["AvgMonthlySpend"] = X["TotalCharges"] / (X["tenure"] + 1)
        X["IsNewCustomer"] = (X["tenure"] < 3).astype(int)
        X["LifetimeValueEstimate"] = X["MonthlyCharges"] * X["tenure"]
        X["TenureGroup"] = pd.qcut(
            X["tenure"], q=4, labels=["Q1", "Q2", "Q3", "Q4"]
        )
        X["MonthlyChargeTier"] = pd.qcut(
            X["MonthlyCharges"], q=4, labels=["Low", "Med", "High", "Very High"]
        )
        return X

# 2. Cargar datos
churn = pd.read_csv('data/extracted/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# AGREGAR ESTO:
# Convertir TotalCharges a numérico (puede tener espacios vacíos)
churn['TotalCharges'] = pd.to_numeric(churn['TotalCharges'], errors='coerce')

# Eliminar filas con valores faltantes
churn = churn.dropna()

# Convertir tenure a numérico si es necesario
churn['tenure'] = pd.to_numeric(churn['tenure'], errors='coerce')
churn['MonthlyCharges'] = pd.to_numeric(churn['MonthlyCharges'], errors='coerce')

# 3. Separar features y target
X = churn.drop('Churn', axis=1)
y = churn['Churn'].map({'Yes': 1, 'No': 0})

# 4. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Detectar columnas
tmp = FeatureAdder().fit_transform(X_train)
num_attribs = tmp.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_attribs = tmp.select_dtypes(exclude=["int64", "float64"]).columns.tolist()
binary_num_attribs = [col for col in num_attribs if tmp[col].dropna().nunique() == 2]
num_attribs = [col for col in num_attribs if col not in binary_num_attribs]

print("Columnas detectadas:")
print(f"Numéricas: {num_attribs}")
print(f"Binarias: {binary_num_attribs}")
print(f"Categóricas: {cat_attribs}")

# 6. Pipeline de preprocessing
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

binary_num_pipeline = "passthrough"

cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# 7. Modelo con tus hiperparámetros reales
best_hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=3,
    max_iter=200,
    max_leaf_nodes=15,
    min_samples_leaf=20,
    random_state=42
)

# 8. Pipeline completo
full_pipeline = Pipeline([
    ("feature_adder", FeatureAdder()),
    ("preprocessor", ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("binary_num", binary_num_pipeline, binary_num_attribs),
        ("cat", cat_pipeline, cat_attribs),
    ])),
    ("classifier", best_hgb)
])

# 9. Entrenar
print("\nEntrenando modelo...")
full_pipeline.fit(X_train, y_train)

# 10. Evaluar
y_pred = full_pipeline.predict(X_test)

print("\n" + "="*50)
print("RESULTADOS:")
print("="*50)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")

# 11. Guardar modelo
joblib.dump(full_pipeline, 'models/best_hgb_azure_azure.pkl')
print("\n✅ Modelo guardado en: models/best_hgb_azure_azure.pkl")

Columnas detectadas:
Numéricas: ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend', 'LifetimeValueEstimate']
Binarias: ['SeniorCitizen', 'IsNewCustomer']
Categóricas: ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TenureGroup', 'MonthlyChargeTier']

Entrenando modelo...

RESULTADOS:
Accuracy: 0.7925
Precision: 0.6289
Recall: 0.5348
F1-Score: 0.5780

✅ Modelo guardado en: models/best_hgb_azure_azure.pkl


In [2]:
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

In [5]:
import json 

# Guardar metricas
with open('models/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

# Guardar hiperparametros
hyperparams = {
    'learning_rate':0.05,
    'max_depth':3,
    'max_iter':200,
    'max_leaf_nodes':15,
    'min_samples_leaf':20,
    'random_state':42
}

with open('models/hyperparams.json', 'w') as f:
    json.dump(hyperparams, f, indent=2)
    